#### ***Conditional Chain***

##### ***The next step is depending on the result of previous step***

In [34]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API Key is Set.")
else:
    print("API Key is not Set.")

API Key is Set.


In [35]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000267A61014F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000267A5E7A1E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [36]:
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser

class Feedback(BaseModel):
    sentiment:str = Field(description = "Positve or Negative")


parser2 = PydanticOutputParser(pydantic_object=Feedback)

format_instructions = parser2.get_format_instructions()

In [37]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template_1 = ChatPromptTemplate.from_messages([
    ("system","You are an Helpful Assistant"),
    ("human","""Analyze the {text}, generate the response.
    {format_instructions}""")
])



In [38]:
classifier_chain = prompt_template_1 | llm | parser2

In [39]:
prompt2 = ChatPromptTemplate.from_messages([
    ("system","You are helpful AI Assistant"),
    ("human","Generate a appriorate response based on Positive Feedback of {text}")
])

In [40]:
prompt3 = ChatPromptTemplate.from_messages([
    ("system","You are helpful AI Assistant"),
    ("human","Generate a appriorate response based on Negative Feedback of {text} without extra text")
])

In [41]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [42]:
from langchain_core.runnables import RunnableBranch,RunnableLambda

branch_chain = RunnableBranch(
    (lambda x:x.sentiment=="Positive",prompt2|llm |parser),
    (lambda x:x.sentiment=="Negative",prompt3|llm |parser),
    RunnableLambda(lambda x:"No Valid feedback found")
)


In [43]:
final_chain = classifier_chain | branch_chain

response = final_chain.invoke({"text":"The heavy rain spoiled our plans for an outdoor picnic.",
"format_instructions":parser2.get_format_instructions()})
print(response)

We’re sorry to hear about your experience and appreciate your feedback; we’ll work to resolve the issue promptly.
